# QueryTheLogs — Inspect Parquet Output

This notebook demonstrates how to query the Parquet files written by the
`parquet-writer` service after running the StreamingLocally pipeline.

**Prerequisites:**
- The stack has been running and messages have flowed through Flink
- Parquet files exist in `./parquet-output/` on the host

**Install dependencies (if running outside Docker):**
```bash
pip install duckdb pyarrow pandas
```

In [ ]:
# Use DuckDB to query Parquet files with plain SQL — no Spark cluster needed.
# DuckDB reads Parquet natively and is ideal for local exploration.

import duckdb
from pathlib import Path

# Directory where parquet-writer drops files (bind-mounted from Docker)
PARQUET_DIR = Path("./parquet-output")

# List all Parquet files currently on disk
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} Parquet file(s):")
for f in parquet_files:
    print(f"  {f.name}")

In [ ]:
# Query all processed (valid) events across every processed_*.parquet file.
# Adjust the glob pattern to target DLQ files: raw-events-dlq_*.parquet

if parquet_files:
    con = duckdb.connect()

    # DuckDB can query a glob of Parquet files as a single virtual table
    result = con.sql("""
        SELECT *
        FROM read_parquet('./parquet-output/processed_*.parquet')
        LIMIT 20
    """)
    result.show()
else:
    print("No Parquet files found. Start the stack and produce some events first.")

In [ ]:
# Inspect dead-letter (DLQ) records — these are events that failed Flink validation.
# Typical columns: raw (original payload string), error (exception message)

dlq_files = list(PARQUET_DIR.glob("raw-events-dlq_*.parquet"))
if dlq_files:
    con = duckdb.connect()
    result = con.sql("""
        SELECT *
        FROM read_parquet('./parquet-output/raw-events-dlq_*.parquet')
        LIMIT 20
    """)
    result.show()
else:
    print("No DLQ Parquet files found yet.")

In [ ]:
# Aggregate summary: count records per topic and compute basic stats on valid values

if parquet_files:
    con = duckdb.connect()

    # Summary of processed (valid) events
    print("=== Processed events summary ===")
    con.sql("""
        SELECT
            COUNT(*)          AS total_records,
            AVG(value)        AS avg_value,
            MIN(value)        AS min_value,
            MAX(value)        AS max_value
        FROM read_parquet('./parquet-output/processed_*.parquet')
    """).show()

    # Count DLQ records if any exist
    if dlq_files:
        print("=== DLQ events summary ===")
        con.sql("""
            SELECT COUNT(*) AS dlq_records
            FROM read_parquet('./parquet-output/raw-events-dlq_*.parquet')
        """).show()